# Leaf stomatal optimization
 

In [ ]:
# | default_exp leaf_stomatal_optimization

In [ ]:
# | hide
from fastcore import *
from nbdev.showdoc import *

In [ ]:
# | export
from plant_hydraulics.utils import brent_root
from plant_hydraulics.leaf_temperature import leaf_temperature
from plant_hydraulics.leaf_boundary_layer import leaf_boundary_layer
from plant_hydraulics.leaf_photosynthesis import leaf_photosynthesis
from plant_hydraulics.leaf_water_potential import leaf_water_potential
from plant_hydraulics.leaf_stomatal_efficiency import leaf_stomatal_efficiency
from plant_hydraulics.parameter_classes import PhysCon, Leaf, Flux, Atmos

In [ ]:
# | export


def leaf_stomatal_optimization(
    physcon: PhysCon, atmos: Atmos, leaf: Leaf, flux: Flux
) -> Flux:
    """
    Leaf stomatal optimization

    Leaf temperature, energy fluxes, photosynthesis, and stomatal conductance
    with water-use efficiency stomatal optimization.
    """

    # Low and high initial estimates for gs (mol H2O/m2/s) ----------------------
    gs1 = 0.002
    gs2 = 2.0

    # Check for minimum stomatal conductance ------------------------------------
    # Linked to low light or drought stress based on the water-use efficiency
    # and cavitation checks for gs1 and gs2 (check1, check2)

    flux, check1 = leaf_stomatal_efficiency(physcon, atmos, leaf, flux, gs1)
    flux, check2 = leaf_stomatal_efficiency(physcon, atmos, leaf, flux, gs2)

    if check1 * check2 < 0:
        # Calculate gs using the function leaf_stomatal_efficiency to iterate gs
        # to an accuracy of tol (mol H2O/m2/s)

        tol = 0.004
        flux, root = brent_root(
            leaf_stomatal_efficiency, physcon, atmos, leaf, flux, gs1, gs2, tol
        )
        flux.gs = root
    else:
        # Low light or drought stress: set gs to minimum conductance
        flux.gs = 0.002

    # Leaf fluxes for this gs
    flux = leaf_boundary_layer(physcon, atmos, leaf, flux)
    flux = leaf_temperature(physcon, atmos, leaf, flux)
    flux = leaf_photosynthesis(physcon, atmos, leaf, flux)
    flux = leaf_water_potential(physcon, leaf, flux)

    return flux

#### Example leaf_stomatal_optimization()